# Train a model on Colab

Runs `python -m src.train` on a Colab GPU with the data, checkpoints and results on Google Drive.

**Resuming:** checkpoints are saved to Drive every 10 minutes, at every epoch end and when you
press Stop. If the run is cut off for any reason, just do **Runtime > Run all** again: it
continues from the last checkpoint (it never starts over, and a finished run is not repeated).
If training crashes while the runtime is still alive, cell 7 restarts it from the checkpoint by
itself.

Drive must contain (see `DRIVE_ROOT` below):
```
who_wrote_this/data/splits/train.parquet
who_wrote_this/data/splits/val.parquet
who_wrote_this/results/class_weights.json
```

In [ ]:
# Settings
MODEL = "transformer"            # transformer | lstm | cnn | mlp
MODE = "train"                   # train: one run with the current config
                                 # tune: every run of tuning.<MODEL> in configs/config.yaml
RUN_NAME = "default"             # MODE = "train" only
BRANCH = "main"
RUN_QUICK_CHECK = False          # True: train on 2,000 rows first (about 2-3 minutes)
MAX_ATTEMPTS = 5                 # automatic restarts after a crash (cell 7)
DRIVE_ROOT = "/content/drive/MyDrive/who_wrote_this"
REPO_URL = "https://github.com/VEDITHREDDY26/Who_Wrote_this.git"
REPO_DIR = "/content/Who_Wrote_this"

In [ ]:
# 1. GPU check (Runtime > Change runtime type > A100 / L4 / T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 3. Code: clone once, then always update to the latest commit of BRANCH
import os

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log --oneline -1

In [ ]:
# 4. Packages (Colab already has torch with CUDA)
!pip install -q -r requirements.txt

In [ ]:
# 5. Paths on Drive, then check that the data is there
from pathlib import Path

os.environ["TD_DATA_DIR"] = f"{DRIVE_ROOT}/data"
os.environ["TD_RESULTS_DIR"] = f"{DRIVE_ROOT}/results"
os.environ["TD_CHECKPOINT_DIR"] = f"{DRIVE_ROOT}/checkpoints"
needed = [f"{DRIVE_ROOT}/data/splits/train.parquet", f"{DRIVE_ROOT}/data/splits/val.parquet",
          f"{DRIVE_ROOT}/results/class_weights.json"]
missing = [p for p in needed if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Upload these to Google Drive first: {missing}")
!python scripts/check_env.py
from src.data.splits import fingerprint_ids, load_split  # noqa: E402

for split in ("train", "val"):
    ids = load_split(split, columns=["id"])["id"]
    print(split, len(ids), fingerprint_ids(ids))
# expected: train 153459 012a901cb6375b61 / val 17101 6c517d46f22dda76

In [ ]:
# 6. Optional quick check, and where the full run will start from
import json

if RUN_QUICK_CHECK:
    !python -m src.train --model {MODEL} --subset 2000 --max-epochs 1 --run-name colab-smoke

if MODE == "tune":
    print("Tuning sweep: finished runs are skipped, unfinished ones resume.")
run_dir = Path(f"{DRIVE_ROOT}/checkpoints/{MODEL}/{RUN_NAME}")
hf_states = sorted(run_dir.glob("checkpoint-*/trainer_state.json"),
                   key=lambda p: int(p.parent.name.split("-")[1]))
if (run_dir / "done.json").exists():
    print("This run already finished; cell 7 will only print the results.")
elif hf_states:
    state = json.loads(hf_states[-1].read_text())
    print(f"RESUMING from {hf_states[-1].parent.name}: epoch {state['epoch']:.2f}, "
          f"step {state['global_step']}/{state['max_steps']}")
elif (run_dir / "last.pt").exists():
    print(f"RESUMING from {run_dir}/last.pt (if that run already finished, cell 7 says so)")
else:
    print("Starting a new run.")

In [ ]:
# 7. Full training, restarted automatically from the last checkpoint after a crash.
# Exit codes 0 (finished), 2 (config/usage error) and 130 (you pressed Stop) are not retried.
import signal
import subprocess
import sys


def run_streaming(cmd: list[str]) -> int:
    """Run cmd, show its output live in this cell, pass a Stop (interrupt) on to it."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    except KeyboardInterrupt:
        proc.send_signal(signal.SIGINT)  # training saves a checkpoint, then exits
        proc.wait()
        raise

if MODE == "tune":  # src.tune restarts crashed runs itself
    run_streaming([sys.executable, "-m", "src.tune", "--model", MODEL])
else:
    cmd = [sys.executable, "-m", "src.train", "--model", MODEL, "--run-name", RUN_NAME]
    for attempt in range(1, MAX_ATTEMPTS + 1):
        print(f"=== attempt {attempt}/{MAX_ATTEMPTS} ===", flush=True)
        code = run_streaming(cmd)
        if code in (0, 2, 130):
            break
        print(f"Training exited with code {code}; resuming from the last checkpoint...")
    else:
        print("Still failing after all attempts; read the error above.")


In [ ]:
# 8. Results
if MODE == "tune":
    sweep = json.load(open(f"{DRIVE_ROOT}/results/tuning/{MODEL}_tuning.json"))
    print(f"chosen: {sweep['chosen']}")
    for r in sweep["runs"]:
        print(f"  {r['run_name']:12s} macro-F1 {r['val_macro_f1']:.4f}  "
              f"human FPR {r['val_human_fpr']:.3f}  best epoch {r['best_epoch']}  "
              f"{r['overrides']}")
    RUN_NAME = sweep["chosen"]
label = MODEL if RUN_NAME == "default" else f"{MODEL}-{RUN_NAME}"
m = json.load(open(f"{DRIVE_ROOT}/results/{label}_val_metrics.json"))
extra = m["extra"]
print(f"val macro-F1 {m['macro_f1']:.4f}  accuracy {m['accuracy']:.4f}  "
      f"human FPR {m['human_fpr']:.4f}")
print(f"best epoch {extra['best_epoch']}, train time {extra['train_time_seconds'] / 60:.1f} min, "
      f"{extra['num_parameters']:,} parameters")
for name, pc in m["per_class"].items():
    print(f"  {name:8s} P {pc['precision']:.3f}  R {pc['recall']:.3f}  F1 {pc['f1']:.3f}")